<a href="https://colab.research.google.com/github/PauloRadatz/opendss-python-examples/blob/main/presentations/ieee_et_pes_pels_joint_chapter_workshop/03_finding_lowest_voltage_bus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3 — Finding the lowest-voltage bus

Here is a small task that sounds trivial: after solving the power flow, find the bus with the lowest voltage on the feeder.

Doing this in OpenDSS by hand means opening the *Show Voltages LN Nodes* report, scrolling through hundreds of rows, and finding the minimum. On `123bus` that is roughly 280 nodes.

In Python it is five lines.

## Setup

In [1]:
!pip install py-dss-interface
!git clone https://github.com/PauloRadatz/opendss-python-examples.git
%cd opendss-python-examples

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 21.1 MB/s eta 0:00:00
Cloning into 'opendss-python-examples'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (189/189), done.
remote: Compressing objects: 100% (159/159), done.
Receiving objects: 100% (189/189), 1.07 MiB | 4.17 MiB/s, done.
remote: Total 189 (delta 81), reused 93 (delta 23), pack-reused 0 (from 0)
Resolving deltas: 100% (81/81), done.
/content/opendss-python-examples


In [2]:
import pandas as pd
import py_dss_interface

dss_file= "/content/opendss-python-examples/feeder_models/IEEETestCases/123Bus/IEEE123Master.dss"

print(f"Master file: {dss_file}")

Master file: /content/opendss-python-examples/feeder_models/IEEETestCases/123Bus/IEEE123Master.dss


## Compile and solve at peak load

Lowest voltages typically appear under heavy load, so we use the peak condition.

In [3]:
dss = py_dss_interface.DSS()
dss.text(f"compile [{dss_file}]")
dss.text("set loadmult=1.0")
dss.text("solve")

print(f"Converged: {dss.solution.converged}")

Converged: 1


## The five-line answer

`dss.circuit.buses_vmag_pu` returns voltage magnitudes in per unit for every node, and `dss.circuit.nodes_names` returns the names in the same order. We pair them and pick the minimum.

In [4]:
voltages_pu = dss.circuit.buses_vmag_pu
node_names = dss.circuit.nodes_names

min_v = min(voltages_pu)
min_node = node_names[voltages_pu.index(min_v)]
min_bus = min_node.split(".")[0]

print(f"Lowest voltage on the feeder")
print(f"  Bus.Node : {min_node}")
print(f"  Bus only : {min_bus}")
print(f"  Voltage  : {min_v:.4f} pu")

Lowest voltage on the feeder
  Bus.Node : 65.1
  Bus only : 65
  Voltage  : 0.9792 pu


## A bit more context: the 10 lowest-voltage nodes

Once the data is in Python, expanding the question is cheap. The cell below builds a sorted DataFrame and shows the 10 worst nodes.

In [5]:
voltages_df = pd.DataFrame({
    "Node": node_names,
    "Voltage (pu)": voltages_pu,
})

# Drop nodes with zero voltage (they typically correspond to unused phases on single-phase laterals).
voltages_df = voltages_df[voltages_df["Voltage (pu)"] > 0]

voltages_df = voltages_df.sort_values("Voltage (pu)").reset_index(drop=True)
voltages_df.head(10)

,Node,Voltage (pu)
0,65.1,0.979211
1,66.1,0.979450
2,64.1,0.979896
3,63.1,0.980220
4,62.1,0.980826
5,160.1,0.981599
6,60.1,0.981599
7,61.1,0.981599
8,61s.1,0.981599
9,51.1,0.984067


## How many nodes are below 0.98 pu?

On a real feeder under stress this is the question that usually matters. One line of pandas. We use 0.98 pu instead of 0.95 pu here — at peak load on this feeder no node falls below 0.95, so 0.98 gives the example a non-zero answer.

In [6]:
below_limit = voltages_df[voltages_df["Voltage (pu)"] < 0.98]
print(f"Nodes below 0.98 pu: {len(below_limit)} out of {len(voltages_df)}")

Nodes below 0.98 pu: 3 out of 278


## Key takeaways

- A task that is tedious in the OpenDSS standalone — searching a long voltage report for a minimum — is five lines in Python.
- `dss.circuit.buses_vmag_pu` and `dss.circuit.nodes_names` are the simplest building blocks for any voltage-screening study.
- Once the result is in a DataFrame, asking follow-up questions (top 10 worst nodes, count of nodes below a limit, group by feeder section) is essentially free.

This is what we mean by *Python turns OpenDSS into a programmable engineering platform*. The simulation engine does not change. What changes is how easily we can ask questions of the result.

## Additional learning resources

If you would like to continue learning OpenDSS and Python for power-system studies, you can find more educational materials and courses here:

- [OpenDSS courses](https://www.pauloradatz.me/opendss-courses)

## Contact

For questions or follow-up about these materials:

- Paulo Radatz
- Email: [paulo.radatz@gmail.com](mailto:paulo.radatz@gmail.com)
- LinkedIn: [linkedin.com/in/pauloradatz](https://www.linkedin.com/in/pauloradatz/)
- Website: [pauloradatz.me](https://www.pauloradatz.me/)